# Evaluation and failure analysis

This notebook scores a checkpoint on fixed validation windows and inspects routed-expert utilization.

In [ ]:
import sys
from pathlib import Path
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'model').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch
from src.evaluate_checkpoint import load_model
from src.training import FixedTokenWindows, collect_routing_statistics, evaluate_fixed_windows

In [ ]:
checkpoint = project_root / 'checkpoints' / 'latest.pt'
val_file = project_root / 'data' / 'processed' / 'val.bin'
if not checkpoint.is_file() or not val_file.is_file():
    raise FileNotFoundError('Run Module 27 first to create a checkpoint and val.bin')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(str(checkpoint), device)
windows = FixedTokenWindows(val_file, seq_len=128, dtype='<u2')
print('device:', device, 'available windows:', len(windows))

In [ ]:
metrics = evaluate_fixed_windows(model.main_model, windows, batch_size=4, max_windows=128)
metrics

In [ ]:
routing_batches = (batch[:, :-1] for batch in windows.batches(4, max_windows=128))
routing = collect_routing_statistics(model.main_model, routing_batches)
routing

Compare checkpoints with the same fixed windows. Balanced routing is useful, but it is separate from language-modeling quality and throughput.